In [961]:
import pandas as pd
import numpy as np
import difflib

# require pip installs
from thefuzz import process 
from faker import Faker
fake = Faker()

In [962]:
n = 100

names = [fake.first_name() for _ in range(n)]
salaries = np.random.exponential(scale=20000, size=n) + 30000

fd1 = pd.DataFrame({
    "Name":names,
    "Salary":salaries
})
fd1.shape


(100, 2)

## split and concat

In [963]:
fd1_a = fd1.loc[:49]    # slicing is INCLUSIVE here
fd1_b = fd1.loc[50:]

print(fd1_a.shape,
      fd1_b.shape)

fd1_a.tail()

(50, 2) (50, 2)


,Name,Salary
45,Samantha,30800.030923
46,Tim,54861.461502
47,Kelly,37901.225383
48,Joe,33776.773661
49,Kim,43257.269962


In [964]:
fd1_ab = pd.concat([fd1_a,fd1_b])
fd1_ab.shape

(100, 2)

## Merge

* use merge to combine columns from two or more dataframes

In [965]:
# use the same names as before

mu = 70     # mean
sigma = 2.75    # stdev
heights = np.random.normal(loc=mu, scale=sigma, size=n)

fd2 = pd.DataFrame({
    "Name":names,
    "Height":heights
})
fd2.shape

(100, 2)

In [966]:
print(fd1.head())
print("---")
print(fd2.head())

       Name        Salary
0    Thomas  45669.107963
1  Jennifer  32990.951466
2      Sean  34300.507967
3     Tracy  31731.477857
4     Julie  33796.075613
---
       Name     Height
0    Thomas  74.927557
1  Jennifer  74.521001
2      Sean  66.912863
3     Tracy  74.877773
4     Julie  68.980822


in this case, we could just add the column, or concatenate

In [967]:
# # add the column
# fd1['Height'] = fd2['Height']
# fd1.head()

# # concatenate
# pd.concat([fd1,fd2['Height']],axis = 'columns')

but this will only work if the rows line up perfectly

In [968]:
# left_on and right_on are only needed if the columns your using to match have different names
fd_merged = pd.merge(fd1,fd2, left_on="Name", right_on="Name")
fd_merged.head()

,Name,Salary,Height
0,Thomas,45669.107963,74.927557
1,Thomas,45669.107963,71.009577
2,Jennifer,32990.951466,74.521001
3,Sean,34300.507967,66.912863
4,Tracy,31731.477857,74.877773


## fuzzy matching

* this is helpful if you're trying to merge data but the values in the match column don't exactly match
* in this example, our main goal is to add one or more columns from "country_codes" to "essential_indicators"

In [969]:
df = pd.read_csv('data/essential_indicators_messy.csv', index_col=0)

df.columns
list1 = df['Country']
print(list1.shape)
list1.head()

(217,)


0       Afghanistan
1           Albania
2           Algeria
3    American Samoa
4           Andorra
Name: Country, dtype: object

In [970]:
df = pd.read_csv('data/country_codes.csv')#, index_col=0)
df.head()

df.columns
list2 = df['name']
print(list2.shape)
list2.head()

(249,)


0       Afghanistan
1     Åland Islands
2           Albania
3           Algeria
4    American Samoa
Name: name, dtype: object

In [971]:
# count the number of exact matches
matches = list1.isin(list2).sum()
matches

np.int64(181)

In [972]:
# Find names in list1 that aren't in list2
missing = list1[~list1.isin(list2)]
print(len(missing))
missing.head()


36


13        Bahamas, The
23             Bolivia
38     Channel Islands
43    Congo, Dem, Rep,
44         Congo, Rep,
Name: Country, dtype: object

In [973]:
# method to find closest match using difflib
def find_closest_match_difflib(name, choices,cutoff=0.8):

    matches = difflib.get_close_matches(name, choices, n=1, cutoff=cutoff)
    return matches[0] if matches else None

In [974]:
for cutoff in [0.8, 0.7, 0.6, 0.5]:
    m1 = find_closest_match_difflib('Bolivia',list2, cutoff=cutoff)
    print(m1)

None
None
None
Somalia


In [975]:
# get score
def get_score(n1,n2):
    if n1 and n2:
        score = difflib.SequenceMatcher(None,n1,n2 ).ratio()
        return score

get_score('Bolivia', 'Somalia')

0.5714285714285714

## using the Fuzz

In [977]:
# match missing names to cc names using thefuzz
def find_closest_match_fuzz(name, choices):
    match, score = process.extractOne(name, choices)
    return match, score# if score > 80 else None

In [978]:
find_closest_match_fuzz("Bolivia", list2.to_list())

('Bolivia, Plurinational State of', 90)

In [979]:
matches, scores = zip(*[find_closest_match_fuzz(n,list2.to_list()) for n in missing])
# scores = [get_score(n,m) for n,m in zip(missing,matches)]

match_table = pd.DataFrame({
    'Name':missing,
    'Match':matches,
    'Score':scores
})

# match_table.dropna()
match_table.sort_values('Score')

,Name,Match,Score
102,Kosovo,Solomon Islands,54
179,"St, Lucia",Saint Lucia,80
89,"Iran, Islamic Rep,","Iran, Islamic Republic of",81
38,Channel Islands,Cocos (Keeling) Islands,86
105,Lao PDR,Lao People's Democratic Republic,86
170,Slovak Republic,Central African Republic,86
125,"Micronesia, Fed, Sts,","Micronesia, Federated States of",86
104,Kyrgyz Republic,Central African Republic,86
207,"Venezuela, RB","Venezuela, Bolivarian Republic of",86
210,West Bank and Gaza,"Bonaire, Sint Eustatius and Saba",86
